# Lab 8. ARIMA Models and Differencing

[Open this lab in Google Colab](https://colab.research.google.com/github/wanghemath/Book-MachineLearning2sda/blob/main/labs/chapter-08-arima-differencing-lab.ipynb)

This lab is designed for **independent study**. It includes the background ideas before each programming step. You should be able to read the explanations, run the code, answer the checkpoint questions, and modify the examples without needing extra instructions.

## Learning goals

By the end of this lab, you should be able to:

1. Explain why a nonstationary series often needs transformation or differencing before ARMA modeling.
2. Simulate and recognize a random walk as an ARIMA(0,1,0) process.
3. Use first and second differences to remove deterministic trends.
4. Understand the danger of over-differencing.
5. Fit ARIMA models in Python using `statsmodels`.
6. Use AIC/BIC, residual ACF, and the Ljung-Box test to compare fitted models.
7. Produce forecasts and prediction intervals on the original data scale.
8. Use AI tools to critique ARIMA modeling choices without blindly trusting them.

## Google Colab notation note

The notebook uses simple LaTeX notation that displays reliably in Google Colab.

## 1. Background: from ARMA to ARIMA

In earlier labs, we modeled **stationary** time series using AR, MA, and ARMA models. A stationary series has roughly stable mean, stable variance, and a dependence structure that does not change over time.

Many real series are not stationary. For example, a price series may wander upward or downward, a population series may grow over time, and a monthly sales series may have a trend and seasonality.

The ARIMA idea is simple:

1. Start with a possibly nonstationary series $X_t$.
2. Difference it $d$ times:

$$
Y_t = (1-B)^d X_t.
$$

3. Model the differenced series $Y_t$ as an ARMA($p,q$) process.

So, $X_t$ is called ARIMA($p,d,q$) when $Y_t=(1-B)^d X_t$ is ARMA($p,q$).

The first difference is

$$
\nabla X_t = X_t - X_{t-1}.
$$

The second difference is

$$
\nabla^2 X_t = \nabla(\nabla X_t) = X_t - 2X_{t-1} + X_{t-2}.
$$

A random walk is the simplest example:

$$
X_t = X_{t-1} + W_t.
$$

Then

$$
\nabla X_t = W_t,
$$

so a random walk is ARIMA(0,1,0).

## 2. Setup

Run the following cell first. It imports the packages and defines helper functions used throughout the lab.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller

rng = np.random.default_rng(7339)

plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True


def plot_series(y, title="Time series", ylabel="value"):
    # Plot a one-dimensional time series.
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(np.arange(len(y)), y)
    ax.set_title(title)
    ax.set_xlabel("time")
    ax.set_ylabel(ylabel)
    plt.show()


def difference(y, d=1):
    # Return the d-th ordinary difference of a one-dimensional array.
    out = np.asarray(y, dtype=float)
    for _ in range(d):
        out = np.diff(out)
    return out


def sample_acf(x, max_lag=30):
    # Compute the sample autocorrelation function from scratch.
    x = np.asarray(x, dtype=float)
    x_centered = x - x.mean()
    denom = np.sum(x_centered ** 2)
    acf_values = []
    for h in range(max_lag + 1):
        num = np.sum(x_centered[h:] * x_centered[: len(x_centered) - h])
        acf_values.append(num / denom)
    return np.array(acf_values)


def plot_acf_from_scratch(x, max_lag=30, title="Sample ACF"):
    acf_values = sample_acf(x, max_lag=max_lag)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.axhline(0, linewidth=1)
    ax.stem(range(max_lag + 1), acf_values, basefmt=" ")
    ax.set_title(title)
    ax.set_xlabel("lag")
    ax.set_ylabel("sample ACF")
    plt.show()
    return acf_values


def adf_summary(y, name="series"):
    # Run the Augmented Dickey-Fuller test and print a compact summary.
    # Small p-values give evidence against a unit root. This is not a complete
    # proof of stationarity, but it is a useful diagnostic.
    stat, pvalue, used_lag, nobs, crit, icbest = adfuller(np.asarray(y), autolag="AIC")
    print(f"ADF test for {name}")
    print(f"  test statistic: {stat: .3f}")
    print(f"  p-value:        {pvalue: .4f}")
    print(f"  used lags:      {used_lag}")
    print(f"  observations:   {nobs}")
    print("  rough interpretation:", "likely stationary" if pvalue < 0.05 else "unit-root behavior not rejected")

## 3. First differences and deterministic trend

Suppose

$$
X_t = \beta_0 + \beta_1 t + Y_t,
$$

where $Y_t$ is stationary noise. The first difference is

$$
\nabla X_t = \beta_1 + Y_t - Y_{t-1}.
$$

This removes the intercept and converts the linear trend into a constant mean. The result may be much closer to stationarity.

In this first example, we simulate a line plus stationary noise and compare the raw series with its first difference.

In [ ]:
n = 250
t = np.arange(n)

beta0 = 5.0
beta1 = 0.08
noise = rng.normal(0, 1.0, size=n)

x_trend = beta0 + beta1 * t + noise
dx_trend = difference(x_trend, d=1)

plot_series(x_trend, title="Linear trend plus noise: raw series")
plot_series(dx_trend, title="First difference: trend mostly removed", ylabel="change")

In [ ]:
print("Raw series mean, first 50 observations:  ", x_trend[:50].mean().round(3))
print("Raw series mean, last 50 observations:   ", x_trend[-50:].mean().round(3))
print("Differenced mean, first 50 observations:", dx_trend[:50].mean().round(3))
print("Differenced mean, last 50 observations: ", dx_trend[-50:].mean().round(3))

### Checkpoint 1

Answer in words before moving on.

1. Why does the raw series not look stationary?
2. What changed after taking the first difference?
3. Does differencing recover the original noise exactly? Why or why not?

## 4. A random walk as ARIMA(0,1,0)

A random walk is defined by

$$
X_t = X_{t-1} + W_t,
$$

where $W_t$ is white noise. The level $X_t$ is not stationary because its variance grows with time. But the first difference is exactly the innovation:

$$
X_t - X_{t-1} = W_t.
$$

So the differenced series is white noise.

In [ ]:
n = 400
w = rng.normal(0, 1.0, size=n)
x_rw = np.cumsum(w)
dx_rw = difference(x_rw, d=1)

plot_series(x_rw, title="Random walk: nonstationary level")
plot_series(dx_rw, title="First difference of random walk: approximately white noise", ylabel="change")

In [ ]:
plot_acf_from_scratch(x_rw, max_lag=40, title="ACF of random walk level: slow decay")
plot_acf_from_scratch(dx_rw, max_lag=40, title="ACF of first difference: near white noise")

In [ ]:
adf_summary(x_rw, name="random walk level")
print()
adf_summary(dx_rw, name="first difference of random walk")

### Checkpoint 2

The ACF of a random walk level often decays slowly. The ACF of its first difference should be much closer to zero after lag 0.

Explain why this agrees with the identity $\nabla X_t = W_t$.

## 5. Why over-differencing is dangerous

Differencing can help, but too much differencing can introduce unnecessary dependence.

For example, if $X_t=W_t$ is already white noise, then

$$
\nabla X_t = W_t - W_{t-1}.
$$

This is no longer white noise. It is an MA(1)-type process with negative lag-1 autocorrelation. This is one reason we should not automatically difference every series.

In [ ]:
w2 = rng.normal(0, 1.0, size=400)
dw2 = difference(w2, d=1)

plot_series(w2, title="White noise: already stationary")
plot_series(dw2, title="First difference of white noise: over-differenced series", ylabel="change")

plot_acf_from_scratch(w2, max_lag=30, title="ACF of white noise")
plot_acf_from_scratch(dw2, max_lag=30, title="ACF after unnecessary differencing")

### Checkpoint 3

Look at the lag-1 autocorrelation after differencing white noise. Is it positive, negative, or near zero? Why does this suggest that over-differencing can create artificial dependence?

## 6. Simulating an ARIMA(1,1,1) process

An ARIMA(1,1,1) process becomes ARMA(1,1) after first differencing.

Let

$$
Y_t = \nabla X_t.
$$

We simulate

$$
Y_t = \phi Y_{t-1} + W_t + \theta W_{t-1},
$$

and then build

$$
X_t = X_{t-1} + Y_t.
$$

The level $X_t$ is nonstationary, but the difference $Y_t$ is stationary if $|\phi|<1$.

In [ ]:
def simulate_arma11(n=400, phi=0.6, theta=0.5, sigma=1.0, burnin=200, seed=7339):
    local_rng = np.random.default_rng(seed)
    total = n + burnin
    w = local_rng.normal(0, sigma, size=total)
    y = np.zeros(total)
    for i in range(1, total):
        y[i] = phi * y[i - 1] + w[i] + theta * w[i - 1]
    return y[burnin:]

phi_true = 0.6
theta_true = 0.5

y_diff = simulate_arma11(n=400, phi=phi_true, theta=theta_true, seed=2026)
x_arima = np.cumsum(y_diff)

plot_series(x_arima, title="Simulated ARIMA(1,1,1): level")
plot_series(y_diff, title="First difference: ARMA(1,1)-like stationary series", ylabel="change")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(x_arima, lags=35, ax=axes[0])
axes[0].set_title("ACF: ARIMA level")
plot_acf(y_diff, lags=35, ax=axes[1])
axes[1].set_title("ACF: first difference")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_pacf(x_arima, lags=35, ax=axes[0], method="ywm")
axes[0].set_title("PACF: ARIMA level")
plot_pacf(y_diff, lags=35, ax=axes[1], method="ywm")
axes[1].set_title("PACF: first difference")
plt.tight_layout()
plt.show()

### Checkpoint 4

Compare the ACF of the level and the ACF of the first difference.

1. Which one looks more stationary?
2. Why is it reasonable to fit an ARMA model to the differenced series?
3. Why is this called an ARIMA model for the original level series?

## 7. Fitting ARIMA models with `statsmodels`

The `statsmodels` function `ARIMA(y, order=(p,d,q))` estimates an ARIMA($p,d,q$) model.

Important modeling principle:

- Choose $d$ to make the series approximately stationary.
- Choose $p$ and $q$ to describe the remaining dependence after differencing.
- Use diagnostics to check whether residuals look like white noise.

We will fit a small grid of candidate models and compare AIC and BIC.

In [ ]:
def fit_arima_grid(y, p_values, d_values, q_values):
    rows = []
    for p in p_values:
        for d in d_values:
            for q in q_values:
                try:
                    model = ARIMA(y, order=(p, d, q))
                    result = model.fit()
                    rows.append({
                        "p": p,
                        "d": d,
                        "q": q,
                        "aic": result.aic,
                        "bic": result.bic,
                        "converged": bool(result.mle_retvals.get("converged", True))
                    })
                except Exception as e:
                    rows.append({
                        "p": p,
                        "d": d,
                        "q": q,
                        "aic": np.nan,
                        "bic": np.nan,
                        "converged": False
                    })
    return pd.DataFrame(rows).sort_values("aic").reset_index(drop=True)

candidate_table = fit_arima_grid(
    x_arima,
    p_values=[0, 1, 2],
    d_values=[0, 1, 2],
    q_values=[0, 1, 2]
)

candidate_table.head(10)

In [ ]:
best_order = tuple(candidate_table.loc[0, ["p", "d", "q"]].astype(int))
print("Best AIC order:", best_order)

best_model = ARIMA(x_arima, order=best_order).fit()
print(best_model.summary())

### Checkpoint 5

The data were generated from an ARIMA(1,1,1)-type process. Did AIC choose exactly (1,1,1)?

Small samples, random noise, and model-estimation uncertainty can cause the selected model to differ from the true data-generating model. Model selection is a guide, not a proof.

## 8. Residual diagnostics

A fitted ARIMA model is not finished just because it has a small AIC. We also need to check residuals.

A reasonable model should have residuals that look roughly like white noise:

- no obvious trend,
- roughly stable variance,
- residual ACF mostly inside noise bands,
- Ljung-Box test not strongly rejecting white-noise residuals.

The Ljung-Box statistic tests whether several residual autocorrelations are jointly large.

In [ ]:
resid = best_model.resid

plot_series(resid, title=f"Residuals from ARIMA{best_order}", ylabel="residual")

fig, ax = plt.subplots(figsize=(9, 4))
plot_acf(resid, lags=35, ax=ax)
ax.set_title(f"Residual ACF from ARIMA{best_order}")
plt.show()

lb = acorr_ljungbox(resid, lags=[10, 20, 30], return_df=True)
lb

### Checkpoint 6

Interpret the residual diagnostics.

1. Do the residuals show obvious remaining pattern?
2. Are many residual autocorrelations significant?
3. Are the Ljung-Box p-values small or large?
4. Would you trust this model for forecasting? Explain your answer.

## 9. Forecasting on the original scale

When we fit `ARIMA(x, order=(p,d,q))`, `statsmodels` internally handles differencing and returns forecasts for the original level series $X_t$, not only for the differenced series.

We now fit on a training portion and forecast a held-out test portion.

In [ ]:
train_size = 330
train = x_arima[:train_size]
test = x_arima[train_size:]

forecast_model = ARIMA(train, order=best_order).fit()
forecast_result = forecast_model.get_forecast(steps=len(test))
forecast_mean = forecast_result.predicted_mean
forecast_ci = forecast_result.conf_int(alpha=0.05)

forecast_index = np.arange(train_size, train_size + len(test))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(np.arange(len(x_arima)), x_arima, label="observed")
ax.axvline(train_size - 1, linestyle="--", label="forecast origin")
ax.plot(forecast_index, forecast_mean, label="forecast")
ax.fill_between(
    forecast_index,
    forecast_ci[:, 0],
    forecast_ci[:, 1],
    alpha=0.2,
    label="95% prediction interval"
)
ax.set_title(f"Train/test forecast with ARIMA{best_order}")
ax.set_xlabel("time")
ax.set_ylabel("level")
ax.legend()
plt.show()

rmse = np.sqrt(np.mean((forecast_mean - test) ** 2))
mae = np.mean(np.abs(forecast_mean - test))
print(f"Test RMSE: {rmse:.3f}")
print(f"Test MAE:  {mae:.3f}")

## 10. Compare with simple baselines

A model is useful only if it improves on simple alternatives. For nonstationary series, two common baselines are:

1. **Naive forecast**: future values equal the last observed value.
2. **Drift forecast**: future values continue the average past slope.

For a random-walk-like series, the naive forecast can be surprisingly strong.

In [ ]:
def naive_forecast(train, h):
    return np.repeat(train[-1], h)


def drift_forecast(train, h):
    slope = (train[-1] - train[0]) / (len(train) - 1)
    return train[-1] + slope * np.arange(1, h + 1)

h = len(test)
naive_pred = naive_forecast(train, h)
drift_pred = drift_forecast(train, h)

comparison = pd.DataFrame({
    "model": [f"ARIMA{best_order}", "naive", "drift"],
    "RMSE": [
        np.sqrt(np.mean((forecast_mean - test) ** 2)),
        np.sqrt(np.mean((naive_pred - test) ** 2)),
        np.sqrt(np.mean((drift_pred - test) ** 2)),
    ],
    "MAE": [
        np.mean(np.abs(forecast_mean - test)),
        np.mean(np.abs(naive_pred - test)),
        np.mean(np.abs(drift_pred - test)),
    ]
})
comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(np.arange(len(x_arima)), x_arima, label="observed")
ax.axvline(train_size - 1, linestyle="--", label="forecast origin")
ax.plot(forecast_index, forecast_mean, label=f"ARIMA{best_order}")
ax.plot(forecast_index, naive_pred, label="naive")
ax.plot(forecast_index, drift_pred, label="drift")
ax.set_title("ARIMA forecast compared with baselines")
ax.set_xlabel("time")
ax.set_ylabel("level")
ax.legend()
plt.show()

### Checkpoint 7

Which model has the smallest RMSE? Which has the smallest MAE? If ARIMA is not clearly better than the baselines, what might that tell you about the data and the forecasting horizon?

## 11. Practice: choosing the differencing order

This section gives you three unlabeled series. For each one, decide whether you would start with $d=0$, $d=1$, or $d=2$.

Guiding questions:

1. Does the level have a stable mean?
2. Does the ACF decay quickly or slowly?
3. Does the first difference look stable?
4. Does the second difference look unnecessarily noisy?

In [ ]:
def simulate_ar1(n=250, phi=0.65, sigma=1.0, seed=1):
    local_rng = np.random.default_rng(seed)
    w = local_rng.normal(0, sigma, size=n)
    x = np.zeros(n)
    for i in range(1, n):
        x[i] = phi * x[i - 1] + w[i]
    return x

n = 250
t = np.arange(n)
series_A = simulate_ar1(n=n, phi=0.7, seed=11)
series_B = np.cumsum(rng.normal(0, 1.0, size=n))
series_C = 2 + 0.03 * t + 0.0008 * (t ** 2) + simulate_ar1(n=n, phi=0.4, seed=13)

practice = {"A": series_A, "B": series_B, "C": series_C}

for label, y in practice.items():
    plot_series(y, title=f"Practice series {label}: level")
    plot_acf_from_scratch(y, max_lag=30, title=f"Practice series {label}: ACF of level")
    plot_series(difference(y, 1), title=f"Practice series {label}: first difference", ylabel="change")
    print("-" * 70)

### Your answers

Write your choices here:

- Series A: I would start with $d=$ ___ because ...
- Series B: I would start with $d=$ ___ because ...
- Series C: I would start with $d=$ ___ because ...

Then modify the code above and examine second differences if needed.

## 12. AI-assisted study prompts

Use these prompts with an AI assistant, then verify every claim using the code and diagnostics above.

### Prompt 1: differencing decision

> I have a time series whose level has a slowly decaying ACF and whose first difference has a stable mean and a rapidly decaying ACF. Explain why ARIMA with $d=1$ may be reasonable. Also list two risks of this conclusion.

### Prompt 2: residual diagnosis

> Here is a residual ACF and Ljung-Box table from an ARIMA model. Explain whether the residuals look like white noise. Do not only say yes or no; explain what evidence should be checked.

### Prompt 3: model comparison

> I fit ARIMA(1,1,1), ARIMA(0,1,1), and ARIMA(2,1,0). AIC prefers one model, but the naive forecast has similar test RMSE. What should I report, and what additional checks should I do?

### Important AI warning

AI can suggest models, but it can also hallucinate diagnostics or ignore leakage. Always check:

1. the time plot,
2. ACF/PACF after differencing,
3. residual plots,
4. out-of-sample performance,
5. comparison with simple baselines.

## 13. Mini-project

Choose one of the following.

### Option A: synthetic ARIMA study

1. Simulate an ARIMA(1,1,1) process with different values of $\phi$ and $\theta$.
2. Fit a grid of ARIMA models.
3. Record how often AIC identifies the correct order.
4. Explain why the selected order may vary across simulations.

### Option B: real-data study

Use a real time series, such as monthly airline passengers, sales, or energy demand.

1. Plot the level.
2. Apply a variance-stabilizing transformation if needed.
3. Choose a differencing order.
4. Fit at least three ARIMA models.
5. Compare AIC/BIC, residual diagnostics, and test forecasts.
6. Compare against naive and drift baselines.

### Option C: over-differencing study

1. Simulate white noise and stationary AR(1) data.
2. Difference each series unnecessarily.
3. Compare the ACF before and after differencing.
4. Explain how over-differencing can create artificial negative autocorrelation.

## 14. Lab checklist

Before you finish, make sure you can do the following without looking at the solution:

- [ ] Define first and second differences.
- [ ] Explain why a random walk is ARIMA(0,1,0).
- [ ] Explain why slowly decaying ACF often suggests nonstationarity.
- [ ] Explain why over-differencing can be harmful.
- [ ] Fit an ARIMA model using `statsmodels`.
- [ ] Compare candidate models using AIC/BIC.
- [ ] Use residual ACF and Ljung-Box tests for diagnostics.
- [ ] Compare ARIMA forecasts with naive and drift baselines.
- [ ] Explain why model selection is not the same as proof.